# Q11. End-to-End ML Preprocessing Pipeline (Titanic)
**Context:** Prepare dataset for ML model training

In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
warnings.filterwarnings('ignore')
# Task 1: Load dataset
titanic = sns.load_dataset('titanic')
print('Shape:', titanic.shape)
print('\nColumns:', list(titanic.columns))
titanic.head()

Shape: (891, 15)

Columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## Step 1: Inspect Missing Values

In [3]:
print('Missing values per column:')
missing = titanic.isnull().sum()
missing_pct = (titanic.isnull().mean() * 100).round(1)
summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(summary[summary['Missing Count'] > 0])

Missing values per column:
             Missing Count  Missing %
age                    177       19.9
embarked                 2        0.2
deck                   688       77.2
embark_town              2        0.2


## Step 2: Select Features & Handle Missing Values

In [8]:
# Select relevant features for model
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
target   = 'survived'

df = titanic[features + [target]].copy()
print('Before missing value handling:')
print(df.isnull().sum())

# Task 2: Handle missing values
df['age'].fillna(df['age'].median(), inplace=True)          
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)  
df['fare'].fillna(df['fare'].median(), inplace=True)

print('\nAfter missing value handling:')
print(df.isnull().sum())

Before missing value handling:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

After missing value handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


## Step 3: Encode Categorical Variables

In [9]:
# Task 3: Encode 'sex' and 'embarked'

# Option A: Manual mapping (transparent)
df['sex_encoded'] = df['sex'].map({'male': 0, 'female': 1})

# Option B: LabelEncoder for embarked
le = LabelEncoder()
df['embarked_encoded'] = le.fit_transform(df['embarked'])

print('Encoding reference:')
print('  sex     : male=0, female=1')
print('  embarked:', dict(zip(le.classes_, le.transform(le.classes_))))
print()

# Drop original categorical columns
df.drop(columns=['sex', 'embarked'], inplace=True)
print('After encoding:')
df.head()

Encoding reference:
  sex     : male=0, female=1
  embarked: {'C': np.int64(0), 'Q': np.int64(1), 'S': np.int64(2)}

After encoding:


,pclass,age,sibsp,parch,fare,survived,sex_encoded,embarked_encoded
0,3,22.0,1,0,7.2500,0,0,2
1,1,38.0,1,0,71.2833,1,1,0
2,3,26.0,0,0,7.9250,1,1,2
3,1,35.0,1,0,53.1000,1,1,2
4,3,35.0,0,0,8.0500,0,0,2


## Step 4: Normalize Numerical Features

In [10]:
# Task 4: Normalize numerical features using MinMaxScaler → range [0, 1]
num_cols = ['age', 'fare', 'sibsp', 'parch', 'pclass']

print('Before normalization (sample stats):')
print(df[num_cols].describe().round(2))

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print('\nAfter normalization (sample stats):')
print(df[num_cols].describe().round(3))

Before normalization (sample stats):
          age    fare   sibsp   parch  pclass
count  891.00  891.00  891.00  891.00  891.00
mean    29.36   32.20    0.52    0.38    2.31
std     13.02   49.69    1.10    0.81    0.84
min      0.42    0.00    0.00    0.00    1.00
25%     22.00    7.91    0.00    0.00    2.00
50%     28.00   14.45    0.00    0.00    3.00
75%     35.00   31.00    1.00    0.00    3.00
max     80.00  512.33    8.00    6.00    3.00

After normalization (sample stats):
           age     fare    sibsp    parch   pclass
count  891.000  891.000  891.000  891.000  891.000
mean     0.364    0.063    0.065    0.064    0.654
std      0.164    0.097    0.138    0.134    0.418
min      0.000    0.000    0.000    0.000    0.000
25%      0.271    0.015    0.000    0.000    0.500
50%      0.347    0.028    0.000    0.000    1.000
75%      0.435    0.061    0.125    0.000    1.000
max      1.000    1.000    1.000    1.000    1.000


## Step 5: Convert to NumPy Array — Final Feature Matrix

In [11]:
# Task 5: Convert to NumPy array
feature_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded', 'embarked_encoded']

X = df[feature_cols].values   # Feature matrix
y = df[target].values         # Target vector

print('=== Final ML-Ready Feature Matrix ===')
print(f'X shape  : {X.shape}')
print(f'y shape  : {y.shape}')
print(f'X dtype  : {X.dtype}')
print(f'y dtype  : {y.dtype}')
print(f'\nX[0:5]:')
print(X[:5])
print(f'\ny[0:5]  : {y[:5]}')
print(f'\nFeature order: {feature_cols}')

=== Final ML-Ready Feature Matrix ===
X shape  : (891, 7)
y shape  : (891,)
X dtype  : float64
y dtype  : int64

X[0:5]:
[[1.         0.27117366 0.125      0.         0.01415106 0.
  2.        ]
 [0.         0.4722292  0.125      0.         0.13913574 1.
  0.        ]
 [1.         0.32143755 0.         0.         0.01546857 1.
  2.        ]
 [0.         0.43453129 0.125      0.         0.1036443  1.
  2.        ]
 [1.         0.43453129 0.         0.         0.01571255 0.
  2.        ]]

y[0:5]  : [0 1 1 1 0]

Feature order: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'sex_encoded', 'embarked_encoded']
